# Fine-Tuning Dataset Builder

## What this notebook does

This notebook exports a fine-tuning dataset from the traces collected by ai-core.

**What fine-tuning does:** It teaches the model HOW to respond (style, format, tone).
**What fine-tuning does NOT do:** It does not teach the model what is in your documents. That is what RAG is for.

## When to use this

- You have collected at least 50+ high-quality traces
- You want to lock in a specific response style or format
- You want the model to respond more consistently to your use cases

## Output format

OpenAI fine-tuning JSONL — one JSON object per line:


In [ ]:
# ============================================================
# Cell 1: Setup and configuration
# ============================================================

import os
import sys
import json
from datetime import datetime

# Add the project root to the Python path so we can import app modules
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, ".env"))

# ============================================================
# CONFIGURATION — Edit these variables before running
# ============================================================

# Filter to a specific use case (set to None to export all)
USE_CASE_FILTER = None  # e.g., "default_rag", "risk_analysis"

# Minimum feedback rating to include (1-5)
# We only want high-quality examples in the fine-tuning dataset
MIN_RATING = 4

# Only include traces where the automatic evaluation passed
REQUIRE_PASSING_EVAL = True

# Maximum number of examples to export
MAX_EXAMPLES = 500

# Output file path
OUTPUT_PATH = f"finetune_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"

print("Configuration:")
print(f"  Use case filter: {USE_CASE_FILTER or 'all'}")
print(f"  Minimum rating: {MIN_RATING}/5")
print(f"  Require passing eval: {REQUIRE_PASSING_EVAL}")
print(f"  Max examples: {MAX_EXAMPLES}")
print(f"  Output: {OUTPUT_PATH}")

In [ ]:
# ============================================================
# Cell 2: Connect to the database
# ============================================================

from app.db.supabase import get_supabase_client

# Initialize the Supabase client
# This reads SUPABASE_URL and SUPABASE_SERVICE_KEY from the .env file
db = get_supabase_client()

print("Connected to Supabase successfully")

In [ ]:
# ============================================================
# Cell 3: Fetch traces
# ============================================================

# Build the traces query
trace_query = (
    db.table("traces")
    .select("id, user_query, prompt_text, response_text, use_case, model_provider, created_at")
    .limit(MAX_EXAMPLES * 3)  # Fetch extra so we have room to filter
)

# Apply use_case filter if specified
if USE_CASE_FILTER:
    trace_query = trace_query.eq("use_case", USE_CASE_FILTER)

traces_result = trace_query.execute()
all_traces = {t["id"]: t for t in (traces_result.data or [])}

print(f"Fetched {len(all_traces)} traces from database")

In [ ]:
# ============================================================
# Cell 4: Filter by evaluation results
# ============================================================

passing_trace_ids = set(all_traces.keys())  # Start with all, then filter

if REQUIRE_PASSING_EVAL and all_traces:
    evals_result = (
        db.table("evaluations")
        .select("trace_id, passed, grounding_score")
        .eq("passed", True)
        .in_("trace_id", list(all_traces.keys()))
        .execute()
    )
    passing_trace_ids = {e["trace_id"] for e in (evals_result.data or [])}
    print(f"Traces with passing evaluation: {len(passing_trace_ids)} / {len(all_traces)}")
else:
    print("Skipping evaluation filter")

In [ ]:
# ============================================================
# Cell 5: Filter by user feedback rating
# ============================================================

if all_traces and passing_trace_ids:
    feedback_result = (
        db.table("feedback")
        .select("trace_id, rating")
        .gte("rating", MIN_RATING)
        .in_("trace_id", list(passing_trace_ids))
        .execute()
    )
    approved_trace_ids = {f["trace_id"] for f in (feedback_result.data or [])}
    print(f"Traces with rating >= {MIN_RATING}: {len(approved_trace_ids)}")
else:
    approved_trace_ids = set()
    print("No traces to filter")

# Final set: must pass both evaluation AND feedback filters
qualified_ids = passing_trace_ids & approved_trace_ids
print(f"Qualified traces (both filters): {len(qualified_ids)}")

In [ ]:
# ============================================================
# Cell 6: Format as OpenAI fine-tuning JSONL
# ============================================================

# Get the qualified traces
qualified_traces = [
    all_traces[tid] for tid in qualified_ids
    if tid in all_traces
][:MAX_EXAMPLES]

print(f"Formatting {len(qualified_traces)} examples...")

# Format each trace as an OpenAI chat fine-tuning example
# See: https://platform.openai.com/docs/guides/fine-tuning
examples = []

for trace in qualified_traces:
    example = {
        "messages": [
            {
                "role": "system",
                # System prompt: defines the assistant behavior
                # We use a simplified version — not the full context-augmented prompt
                "content": "You are an expert document analyst. Answer questions accurately based on the provided context.",
            },
            {
                "role": "user",
                # The user query (raw question, not the full assembled prompt)
                "content": trace["user_query"],
            },
            {
                "role": "assistant",
                # The high-quality response we want the model to learn from
                "content": trace["response_text"],
            },
        ]
    }
    examples.append(example)

print(f"Formatted {len(examples)} fine-tuning examples")

In [ ]:
# ============================================================
# Cell 7: Write JSONL file and print statistics
# ============================================================

if examples:
    # Write the JSONL file — one JSON object per line
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(json.dumps(example, ensure_ascii=False) + "
")
    
    print(f"
=== Export Complete ===")
    print(f"File: {OUTPUT_PATH}")
    print(f"Total examples: {len(examples)}")
    
    # Use case distribution
    use_case_counts = {}
    for trace in qualified_traces:
        uc = trace["use_case"]
        use_case_counts[uc] = use_case_counts.get(uc, 0) + 1
    print(f"Use case distribution: {use_case_counts}")
    
    # Average response length
    avg_len = sum(len(t["response_text"].split()) for t in qualified_traces) / len(qualified_traces)
    print(f"Avg response length: {avg_len:.0f} words")
    
    print(f"
To fine-tune with OpenAI CLI:")
    print(f"  openai api fine_tuning.jobs.create -t {OUTPUT_PATH} -m gpt-4o-mini")
else:
    print("No examples to export. Try lowering MIN_RATING or removing REQUIRE_PASSING_EVAL.")